# 1. Visualizing and cleaning stock data

A price chart can be misleading if the data contains bad quotes or inconsistent ticker names. We'll compare five simulated stocks, introduce some errors, and work through how to find and fix them.

The examples build on [pandas basics](../Lecture%202/2_pandas_basics.ipynb) and [portfolio analysis](../Lecture%202/3_pandas_to_portfolio.ipynb). This notebook runs on its own from a fresh kernel; all prices are generated here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
%matplotlib inline
# Supplied setup: locate this repository even if Jupyter starts in a lecture folder.
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Generate some stock data

`generate_stock_data` simulates prices using geometric Brownian motion and adds open, high, low, close, and volume columns. Its drift and volatility inputs are annualized. Keeping the seed fixed gives the same data each time you run it.

Each stock's DataFrame has dates as its index. `concat` combines the stocks into one table, and `reset_index` moves the dates into a column. The dates include weekdays but do not account for exchange holidays.

In [ ]:
def generate_stock_data(ticker, start_price, volatility, drift, days=252, seed=None):
    """Generate synthetic OHLCV; drift and volatility are annualized inputs."""
    rng = np.random.default_rng(seed)
    daily_vol = volatility / np.sqrt(252)
    log_returns = rng.normal((drift - 0.5 * volatility ** 2) / 252, daily_vol, days)
    closes = start_price * np.exp(np.cumsum(log_returns))
    # Positive opens and intraday ranges; every open and close lies inside the high/low.
    opens = closes * np.exp(rng.normal(0, daily_vol * 0.3, days))
    highs = np.maximum(opens, closes) * np.exp(np.abs(rng.normal(0, daily_vol * 0.6, days)))
    lows = np.minimum(opens, closes) * np.exp(-np.abs(rng.normal(0, daily_vol * 0.6, days)))
    dates = pd.date_range(end='2024-12-06', periods=days, freq='B', name='date')
    return pd.DataFrame({'open': opens, 'high': highs, 'low': lows, 'close': closes,
                         'volume': rng.integers(1_000_000, 50_000_000, days),
                         'ticker': ticker}, index=dates)

In [ ]:
stocks = {
    'AAPL': generate_stock_data('AAPL', 185, 0.25, 0.10, seed=42),
    'MSFT': generate_stock_data('MSFT', 420, 0.20, 0.08, seed=43),
    'NVDA': generate_stock_data('NVDA', 125, 0.55, 0.12, seed=44),
    'JPM': generate_stock_data('JPM', 198, 0.22, 0.05, seed=45),
    'XOM': generate_stock_data('XOM', 112, 0.35, 0.04, seed=46),
}
generated_prices = pd.concat(stocks.values()).reset_index()
prices = generated_prices.copy()
display(prices.head())
print('Observations:', len(prices), '| Assets:', prices['ticker'].nunique())

### Calculating returns

A price that moves from 100 to 110 and then to 99 has daily returns of +10% and −10%. The total return is **−1%**: the second day's loss is calculated from 110, not 100.

`shift(1)` puts the previous price beside the current one. We can divide the two prices to calculate a return, or use `pct_change` for the same calculation.

In [ ]:
toy = pd.Series([100.0, 110.0, 99.0], name='price')
previous = toy.shift(1)
display(pd.DataFrame({'price': toy, 'previous': previous, 'return': toy / previous - 1}))
print('Same calculation:', toy.pct_change(fill_method=None).tolist())
print('Whole-period return:', toy.iloc[-1] / toy.iloc[0] - 1)

## 2. Comparing stocks with plots

MSFT has a higher share price than AAPL in this sample. Does that tell us which stock performed better?

Before calculating returns, sort by ticker and date. Then group by ticker so each price is compared with the previous price for the same stock.

In [ ]:
prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)
prices['return'] = prices.groupby('ticker')['close'].pct_change(fill_method=None)
print('The first return for each asset is undefined:')
display(prices.groupby('ticker').head(1)[['ticker', 'date', 'close', 'return']])

The first chart shows prices in dollars. In the second, every stock starts at 100, which makes their percentage changes easier to compare.

`fig` is the whole figure; `axes[0]` and `axes[1]` are the two plots inside it. The stock report later uses the same structure with four plots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
for ticker, rows in prices.groupby('ticker'):
    axes[0].plot(rows['date'], rows['close'], label=ticker)
    normalized = rows['close'] / rows['close'].iloc[0] * 100
    axes[1].plot(rows['date'], normalized, label=ticker)
axes[0].set(title='Share prices', xlabel='Date', ylabel='Price')
axes[1].set(title='Same starting point', xlabel='Date', ylabel='Indexed price (start = 100)')
for ax in axes:
    ax.legend()
plt.show()

### Daily returns and volatility

The price chart shows performance over time. A histogram shows how often different daily returns occurred. We can also compare daily standard deviations and the worst daily return for each stock. A stock can finish the period with a large gain and still have had substantial daily swings.

In [ ]:
example_msft = prices.loc[prices['ticker'] == 'MSFT'].copy()
(example_msft['return'] * 100).plot.hist(bins=20, title='Synthetic MSFT daily returns', xlabel='Daily return (%)')
plt.show()
example_return_summary = prices.loc[prices['ticker'].isin(['AAPL', 'MSFT'])].groupby('ticker')['return'].agg(
    observations='count', daily_vol_pct='std', worst_day_pct='min')
example_return_summary[['daily_vol_pct', 'worst_day_pct']] *= 100
display(example_return_summary.round(2))

## 3. Finding errors in the data

We'll make a copy of the generated data and add missing closes, a decimal error, inconsistent ticker formatting, invalid volumes, and a duplicate row. The original data stays available for comparison.

Watch what the decimal error does to the chart. If the analysis ended on that date, the stock would appear to have made an enormous gain.

In [ ]:
raw = generated_prices.copy()
raw.loc[2, 'ticker'] = ' aapl '
raw.loc[[4, 5, len(stocks['AAPL'])], 'close'] = np.nan
raw.loc[10, 'close'] *= 100
raw['volume'] = raw['volume'].astype(float)
raw.loc[12, 'volume'] = -100
raw.loc[14, 'volume'] = np.inf
raw = pd.concat([raw, raw.iloc[[3]]], ignore_index=True).sample(frac=1, random_state=7)
raw.plot.scatter(x='date', y='close', title='The vendor file: investigate before interpreting', figsize=(10, 3))
plt.show()

### A close below the daily low

A closing price should be between the day's low and high. DEMO_A violates that condition in the example below. A Boolean mask selects the row we need to investigate.

In [ ]:
example_quotes = pd.DataFrame({
    'ticker': ['DEMO_A', 'DEMO_B'],
    'close': [98.0, 105.0], 'low': [99.0, 101.0], 'high': [103.0, 107.0],
})
example_bad_prices = example_quotes.loc[example_quotes['close'] < example_quotes['low'],
                                        ['ticker', 'close', 'low']]
display(example_bad_prices)

In [ ]:
print('Missing values:')
display(raw.isna().sum())
print('Ticker variants:', raw['ticker'].unique())
print('Non-finite volume:', (~np.isfinite(raw['volume'])).sum())
# Infinity is non-finite, but isna() does not count it as missing.

## 4. Cleaning a vendor's ticker names and quotes

A portfolio might use `AAPL` while a vendor sends `aapl`, ` AAPL `, or `<b>AAPL</b>`. Those strings need to match before we can join the quotes to our holdings. A quote such as `"price: 150.25"` also needs to become a number before we can calculate a position's value.

We'll clean the sample below and keep the original columns beside the results. Look for rows that refer to the same stock and quotes that need further investigation.

In [ ]:
vendor_text = pd.DataFrame({
    'raw_symbol': [' nasdaq: aapl ', 'NASDAQ:<b>AAPL</b>', ' nasdaq : msft ', 'NYSE:JPM', None],
    'raw_name': [' Apple   Inc. ', 'APPLE INC.', 'Microsoft\tCorp.', ' JPMORGAN   CHASE ', None],
    'raw_quote': [' price: 150.25 ', 'PRICE :151.00', 'price: -5', 'price: unavailable', None],
})
display(vendor_text)
print('Distinct raw symbols:', vendor_text['raw_symbol'].nunique())

### Cleaning text columns

Pandas' `.str` methods apply a text operation to each value in a column. Here we remove HTML tags, trim whitespace with `strip`, and convert the symbols to uppercase. Then `split(':')` separates the exchange from the ticker.

The company names contain extra spaces and tabs. Replacing `r'\s+'` with a single space fixes those. Converting to pandas' `string` dtype preserves missing values.

Use `regex=True` for patterns such as repeated whitespace, and `regex=False` when replacing literal text.

In [ ]:
example_text = vendor_text.copy()
symbol_text = (example_text['raw_symbol'].astype('string')
               .str.replace(r'<[^>]+>', '', regex=True)
               .str.strip().str.upper())
symbol_parts = symbol_text.str.split(':', n=1, expand=True)
example_text['exchange'] = symbol_parts[0].str.strip()
example_text['ticker'] = symbol_parts[1].str.strip()
example_text['company_name'] = (example_text['raw_name'].astype('string')
                                .str.strip().str.upper()
                                .str.replace(r'\s+', ' ', regex=True))
display(example_text[['raw_symbol', 'exchange', 'ticker', 'raw_name', 'company_name']])
print('Distinct cleaned tickers:', example_text['ticker'].nunique())

### Converting quotes to numbers

`str.extract` extracts the number after `price:`. The pattern accepts spaces, a decimal point, and a positive or negative sign. `pd.to_numeric` converts the result to a numeric column.

With `errors='coerce'`, a value that cannot be converted becomes missing. Keep it missing for review: replacing it with zero would make a holding appear worthless.

We still need to check the converted values. `price: -5` parses successfully, but −5 is not a valid equity price. Matching the whole field also prevents us from accidentally treating a number in a vendor note as a quote.

In [ ]:
price_pattern = r'^price\s*:\s*([+-]?\d+(?:\.\d+)?)$'
quote_text = example_text['raw_quote'].astype('string').str.strip().str.lower()
example_price_text = quote_text.str.extract(price_pattern, expand=False)
example_text['price'] = pd.to_numeric(example_price_text, errors='coerce')
example_text['price_needs_review'] = example_text['price'].isna() | example_text['price'].le(0)
display(example_text[['raw_quote', 'price', 'price_needs_review']])
display(example_text.loc[example_text['price_needs_review']])
assert example_text.loc[0, 'price'] == 150.25
assert example_text.loc[2, 'price'] == -5
assert example_text.loc[[3, 4], 'price'].isna().all()
print('Quotes needing review:', example_text['price_needs_review'].sum())

### Duplicate and conflicting quotes

Both AAPL labels now match, but their prices differ. We would need more information about the sources to choose between them.

For the price-history table, clean ticker names before checking for repeated `(ticker, date)` keys. We can remove exact duplicate rows. If two rows have the same key but different values, we need to investigate the disagreement.

In [ ]:
clean = raw.copy()
clean['ticker'] = clean['ticker'].str.strip().str.upper()
key = ['ticker', 'date']
display(clean.loc[clean.duplicated(key, keep=False)].sort_values(key))
clean = clean.drop_duplicates().sort_values(key).reset_index(drop=True)
assert not clean.duplicated(key).any(), 'Conflicting records need a source-selection rule.'

### Finding missing dates

`.isna()` finds missing values in rows that exist. It cannot tell us that an entire row is absent. For that, we need a list of expected trading dates for the stock.

The example below has a supplied calendar that excludes January 1. We remove January 3 from DEMO_A's data and check that the function finds it. Using all weekdays would incorrectly include the January 1 holiday.

For HW2, you may use each ticker's dates in `ohlc` as the reference calendar. State that assumption in your analysis. Report missing dates without inventing prices for them.

In [ ]:
def missing_sessions(frame, expected_dates):
    """Return expected dates absent from one ticker's observations."""
    observed = pd.DatetimeIndex(frame['date'].dropna().unique())
    return pd.DatetimeIndex(expected_dates).difference(observed).sort_values()

expected_sessions = pd.to_datetime(['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'])
calendar_delivery = pd.DataFrame({'ticker': ['DEMO_A'] * 3, 'date': expected_sessions[[0, 2, 3]]})
gap_dates = missing_sessions(calendar_delivery, expected_sessions)
print(f"DEMO_A is missing {len(gap_dates)} sessions between {expected_sessions.min().date()} and {expected_sessions.max().date()}: {gap_dates.tolist()}")
assert gap_dates.tolist() == [pd.Timestamp('2024-01-03')]

### Collecting the issues in a table

The function below runs several checks and returns a table of the problems it finds. Each mask selects rows with a particular issue, and `concat` combines them. A row may appear more than once if it has several problems. Clean the ticker names and remove exact duplicates before calling the function.

The return check flags `abs(return - mean) > 3 * std`, using each ticker's sample mean and standard deviation. It checks both positive and negative returns. Missing or nonpositive prices cannot give usable returns, and the first return remains missing. If the standard deviation is zero, there are no statistical flags.

This check uses the full history available to us. A large error can inflate the standard deviation enough to hide other errors, so we also check OHLC relationships directly.

In [ ]:
def price_issue_report(frame):
    """Diagnose normalized, unique ticker/date rows; statistical flags require review."""
    data = frame.sort_values(['ticker', 'date']).copy()
    price_columns = ['open', 'high', 'low', 'close']
    valid_close = data['close'].where(np.isfinite(data['close']) & data['close'].gt(0))
    returns = valid_close.groupby(data['ticker']).pct_change(fill_method=None)
    grouped = returns.groupby(data['ticker'])
    mean, std = grouped.transform('mean'), grouped.transform('std')
    masks = {
        'missing_price': data[price_columns].isna().any(axis=1),
        'invalid_price': (~np.isfinite(data[price_columns]) | data[price_columns].le(0)).any(axis=1),
        'ohlc_bounds': (data['high'].lt(data[['open', 'close']].max(axis=1))
                        | data['low'].gt(data[['open', 'close']].min(axis=1))
                        | data['high'].lt(data['low'])),
        'invalid_volume': ~np.isfinite(data['volume']) | data['volume'].lt(0),
        'zero_volume': data['volume'].eq(0),
        'extreme_return': (returns - mean).abs().gt(3 * std) & std.gt(0),
    }
    parts = []
    for issue, mask in masks.items():
        rows = data.loc[mask, ['ticker', 'date']].rename(columns={'date': 'issue_date'})
        rows['issue_type'] = issue
        rows['description'] = 'Investigate ' + issue.replace('_', ' ')
        rows['severity'] = 'medium' if issue in ['zero_volume', 'extreme_return'] else 'high'
        parts.append(rows)
    return pd.concat(parts, ignore_index=True)[['ticker', 'issue_type', 'issue_date', 'description', 'severity']]

quality_report = price_issue_report(clean)
gap_issues = pd.DataFrame({'ticker': 'DEMO_A', 'issue_type': 'missing_session',
    'issue_date': gap_dates, 'description': 'Absent from supplied session calendar', 'severity': 'medium'})
display(quality_report.head(10))
display(gap_issues)
display(quality_report.groupby(['issue_type', 'severity']).size().rename('issues'))

## 5. Repairing bad prices

We know that **only the close was corrupted** in this example, so we can use the open, high, and low to help repair it. With real data, an inconsistent OHLC row doesn't tell us which field is wrong.

Flag large returns for investigation. If a return is unusual but the prices are valid, there may be no reason to change it.

In [ ]:
bad_close = (~np.isfinite(clean['close']) | (clean['close'] <= 0)
             | (clean['close'] < clean['low']) | (clean['close'] > clean['high']))
clean['close_repaired'] = bad_close
clean.loc[bad_close, 'close'] = np.nan
print('Close values to investigate:', bad_close.sum())

### When forward fill fails

Yesterday's close is outside today's range in the example below. Forward filling would remove the missing value but leave an impossible OHLC row.

We have today's high and low when reviewing this data. Those values would not have been available at the start of the trading day.

In [ ]:
gap_example = pd.DataFrame({'low': [99.0, 109.0], 'high': [101.0, 111.0], 'close': [100.0, np.nan]})
gap_example['ffill_candidate'] = gap_example['close'].ffill()
gap_example['candidate_in_range'] = gap_example['ffill_candidate'].between(gap_example['low'], gap_example['high'])
gap_example

We'll use the previous close if it falls within today's range. Otherwise, we'll use the midpoint of today's high and low. Both choices are estimates, so we keep a flag for each repaired value.

The code first calculates a candidate value, checks whether it is in range, and chooses the replacement. Backward fill and interpolation use later observations; they are unsuitable for a feature that needs to have been available earlier.

In [ ]:
previous = clean.groupby('ticker')['close'].ffill()
within_range = previous.between(clean['low'], clean['high'])
midpoint = (clean['high'] + clean['low']) / 2
replacement = previous.where(within_range, midpoint)
clean['close'] = clean['close'].fillna(replacement)
invalid_volume = ~np.isfinite(clean['volume']) | (clean['volume'] < 0)
clean['volume_repaired'] = invalid_volume
clean.loc[invalid_volume, 'volume'] = 0  # A flagged placeholder for unknown volume, not proof of no trading.
display(clean.loc[clean['close_repaired'] | clean['volume_repaired']])

### Checking the repaired data

An `assert` stops execution if its condition is false. The checks below verify prices, OHLC bounds, volumes, and unique keys. Which one would catch the forward-fill problem above?

A return depends on two closes. Changing one close can therefore change both that day's return and the following day's return.

In [ ]:
price_cols = ['open', 'high', 'low', 'close']
assert np.isfinite(clean[price_cols]).all().all()
assert clean[price_cols].gt(0).all().all()
assert clean['close'].between(clean['low'], clean['high']).all()
assert clean['open'].between(clean['low'], clean['high']).all()
assert np.isfinite(clean['volume']).all() and clean['volume'].ge(0).all()
assert not clean.duplicated(key).any()
clean['return'] = clean.groupby('ticker')['close'].pct_change(fill_method=None)
clean['return_affected'] = clean['close_repaired'] | clean.groupby('ticker')['close_repaired'].shift(1, fill_value=False)
print('Checks passed. We still need to explain the estimates.')

### Other missing values and interpolation

Remove rows with **all four raw OHLC fields missing** before filling. For partial gaps, sort by ticker and date, fill within each ticker, and check the OHLC bounds again. Keep unresolved rows separately with a reason, including leading gaps that have no earlier price to use.

Interpolation estimates a value from neighboring observations. The example below assumes that the vendor has confirmed an error between two valid quotes. `limit_area='inside'` fills that interior gap and leaves the gaps at either end missing. Because it uses a later quote, this repair is appropriate for reviewing historical data, but would not have been possible on the date of the error.

In [ ]:
repair_example = pd.DataFrame({
    'ticker': ['DEMO_C'] * 4 + ['DEMO_D'] * 3,
    'date': pd.to_datetime(['2024-02-01', '2024-02-02', '2024-02-05', '2024-02-06',
                            '2024-02-01', '2024-02-02', '2024-02-05']),
    'open': [100, np.nan, 102, 104, np.nan, 200, 202],
    'high': [102, np.nan, 104, 106, 201, 202, 204],
    'low': [98, np.nan, 100, 102, 197, 198, 200],
    'close': [100, np.nan, 102, 104, np.nan, 200, 202],
})
all_prices_missing = repair_example[['open', 'high', 'low', 'close']].isna().all(axis=1)
retained_example = repair_example.loc[~all_prices_missing].copy()
retained_example[['open', 'high', 'low', 'close']] = retained_example.groupby('ticker')[['open', 'high', 'low', 'close']].ffill()
unresolved_example = retained_example[['open', 'high', 'low', 'close']].isna().any(axis=1)
display(retained_example.loc[unresolved_example])  # DEMO_D's leading gap stays unresolved.
confirmed_error = pd.Series([np.nan, 100.0, np.nan, 104.0, np.nan])
interpolated_example = confirmed_error.interpolate(limit_area='inside')
display(pd.DataFrame({'before': confirmed_error, 'candidate': interpolated_example}))

### Raw and adjusted prices

Use raw open, high, low, and close to check the day's price range. Use adjusted close to calculate performance across splits and distributions. The adjustment factor is `adj_close / close`, provided both prices are reliable.

In the homework, check `adj_close` separately. Forward filling it, or carrying an adjustment factor across a corporate action, can produce incorrect returns. The split example below uses a known factor for each date to reconstruct a missing adjusted close from the raw close.

For HW2, you may obtain adjustment factors from matching `(ticker, ts)` rows in `ohlc`. Document the source, validate raw close, and flag the values you repair. Set aside rows you cannot reasonably repair and report how much data you excluded. Use the reference to investigate and repair individual problems; replacing the whole dirty table would skip the cleaning work.

In [ ]:
adjustment_example = pd.DataFrame({
    'close': [100.0, 50.0, 51.0],
    'adj_close': [50.0, np.nan, 51.0],
    'trusted_factor': [0.5, 1.0, 1.0],
})
adjustment_example['adj_close_repaired'] = adjustment_example['adj_close'].isna()
adjustment_example['adj_close'] = adjustment_example['adj_close'].fillna(
    adjustment_example['close'] * adjustment_example['trusted_factor'])
display(adjustment_example)
print('Raw returns:', adjustment_example['close'].pct_change(fill_method=None).tolist())
print('Adjusted returns:', adjustment_example['adj_close'].pct_change(fill_method=None).tolist())

## 6. Handling missing volume

The next example replaces a missing volume with zero and records the change in a flag. That flag matters: the original volume is unknown, so the replacement does not mean that no shares traded.

In [ ]:
example_volumes = pd.DataFrame({'ticker': ['DEMO_A', 'DEMO_B', 'DEMO_C'],
                               'volume': [100.0, np.nan, 200.0]})
example_volumes['volume_repaired'] = example_volumes['volume'].isna()
example_volumes.loc[example_volumes['volume_repaired'], 'volume'] = 0
display(example_volumes)
print('DEMO_B volume is unknown; its zero is a flagged placeholder.')

### Saving the cleaned data

Save the cleaned table with its repair flags, and keep the original data for comparison. The chart below shows the simulated price histories after our repairs. Any analysis using these prices should explain which values were estimated and how.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
for ticker, rows in clean.groupby('ticker'):
    ax.plot(rows['date'], rows['close'] / rows['close'].iloc[0] * 100, label=ticker)
ax.set(title='After documented repairs', xlabel='Date', ylabel='Indexed price (start = 100)')
ax.legend()
output_dir = course_root / 'Lectures/Lecture 4/outputs'
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / 'clean_prices.png', dpi=150, bbox_inches='tight')
clean.to_csv(output_dir / 'clean_prices_demo.csv', index=False)
plt.show()

## Reviewing the results

Before using the cleaned data, check that you can explain the missing first return, the order of the grouping and sorting steps, and the assumptions behind each repair. Removing nulls and fixing text formatting do not by themselves establish that the prices are accurate.

Continue with [SQL queries](2_sql_and_data_quality.ipynb). [Homework 2](../Lecture%202/hw2.md) is due October 29.

## Exercises

Replace the placeholders with your answers, then run the checks below each exercise.

### Exercise C: compare return distributions

Select NVDA into `your_nvda` and compare its daily returns with JPM's. Use histograms with the same bin edges and horizontal scale, labeled in percent.

Create `your_return_comparison`, indexed by ticker, with three columns for NVDA and JPM: `observations` (the count of nonmissing returns), `daily_vol_pct` (sample standard deviation in percent), and `worst_day_pct` (minimum daily return in percent). Which stock was more volatile in this sample?

Use `.isin`, `groupby`, and `agg`. Multiply fractional statistics by 100 for the table. You can create common histogram bins with `np.linspace` over the combined return range.

For an additional comparison, calculate each stock's total return. Did the stock with the higher return also have greater daily volatility?

In [ ]:
your_nvda = None
your_return_comparison = None
# Select the two assets, compare their return distributions, and calculate the table.

In [ ]:
if your_nvda is None or your_return_comparison is None:
    print('Compare NVDA and JPM distributions and calculate the return summary.')
else:
    assert len(your_nvda) == len(stocks['NVDA']) and your_nvda['ticker'].eq('NVDA').all()
    assert your_nvda['return'].isna().sum() == 1
    assert set(your_return_comparison.index) == {'NVDA', 'JPM'}
    for symbol in ['NVDA', 'JPM']:
        asset_returns = prices.loc[prices['ticker'] == symbol, 'return'].dropna()
        assert your_return_comparison.loc[symbol, 'observations'] == len(asset_returns)
        assert np.isclose(your_return_comparison.loc[symbol, 'daily_vol_pct'], asset_returns.std() * 100)
        assert np.isclose(your_return_comparison.loc[symbol, 'worst_day_pct'], asset_returns.min() * 100)
    print('Summary checks passed. Check common histogram bins, axis limits, and percent labels.')

### Exercise D: find invalid closes

Find every row in `raw` whose close is missing, non-finite, nonpositive, below its low, or above its high. Store the columns `date`, `ticker`, `close`, `low`, and `high` in `your_bad_prices`.

Combine the conditions with `|`. Remember that a comparison such as `close < low` will not catch a missing close. As an extension, add a column describing the problem with each row.

In [ ]:
your_bad_prices = None

In [ ]:
if your_bad_prices is None:
    print('Combine missing-value and OHLC checks into one exception table.')
else:
    expected_bad_mask = (~np.isfinite(raw['close']) | raw['close'].le(0)
                         | raw['close'].lt(raw['low']) | raw['close'].gt(raw['high']))
    expected_bad = raw.loc[expected_bad_mask, ['date', 'ticker', 'close', 'low', 'high']]
    pd.testing.assert_frame_equal(your_bad_prices.sort_index(), expected_bad.sort_index())
    print('Exception-table checks passed. Explain which rule caught each kind of problem.')

### Exercise: repair negative volume

Make a copy of `clean` called `new_delivery` and set its first row's volume to −5. Identify negative and non-finite volumes, flag the affected rows, and replace those volumes with zero.

Assign the repaired table to `your_delivery`. In `your_quality_note`, write a sentence explaining what the zero replacement tells us about the original volume. Check the result without changing `raw` or `clean`.

How would your approach differ if the missing value were a ticker's first price observation?

In [ ]:
new_delivery = clean.copy()
new_delivery.loc[new_delivery.index[0], 'volume'] = -5
your_delivery = None
your_quality_note = ''

In [ ]:
if your_delivery is None:
    print('Complete the new-delivery repair and write a quality note.')
else:
    assert len(your_delivery) == len(clean)
    assert np.isfinite(your_delivery['volume']).all() and your_delivery['volume'].ge(0).all()
    assert your_delivery.loc[your_delivery.index[0], 'volume'] == 0
    assert your_delivery.loc[your_delivery.index[0], 'volume_repaired']
    assert your_quality_note.strip()
    print('Repair checks passed. Explain why the repaired zero is not an observed zero.')

### Exercise: clean another vendor file

The next file contains JPM and NVDA quotes. Keep every row and all three original columns, then add:

1. `exchange` and `ticker`: remove HTML tags, split on `:`, trim whitespace, and convert to uppercase. Preserve missing tickers as missing values.
2. `company_name`: trim and uppercase the names, and replace repeated whitespace with one space.
3. `price`: extract the number from a field in the `price:` format and convert it to a numeric dtype. Leave missing or malformed fields missing.
4. `price_needs_review`: flag missing, malformed, or nonpositive prices.

Store the result in `your_text_clean`. In `your_text_note`, explain which quotes you would exclude from a portfolio valuation and why. You can reuse `price_pattern`, the `.str` methods above, and `pd.to_numeric`.

Check what happens with `batch 7 price:120`: extracting the first number would give the wrong price. You can also add `price: -12.50` to test that your parser preserves the negative sign.

In [ ]:
practice_text = pd.DataFrame({
    'raw_symbol': [' nyse: jpm ', 'NYSE:<i>JPM</i>', ' nasdaq : nvda ', 'NASDAQ:NVDA', None],
    'raw_name': [' JPMorgan   Chase ', 'jpmorgan chase', ' NVIDIA\tCorp. ', 'nvidia  corp.', None],
    'raw_quote': [' PRICE : 210.50 ', 'price:211', 'price:0', 'batch 7 price:120', None],
})
display(practice_text)
your_text_clean = None
your_text_note = ''

In [ ]:
if your_text_clean is None:
    print('Clean the text columns, parse the quotes, and write a note about a flagged row.')
else:
    assert isinstance(your_text_clean, pd.DataFrame)
    assert len(your_text_clean) == len(practice_text), 'Keep all rows so rejected values remain inspectable.'
    pd.testing.assert_frame_equal(your_text_clean[practice_text.columns], practice_text)
    assert your_text_clean['exchange'].iloc[:4].tolist() == ['NYSE', 'NYSE', 'NASDAQ', 'NASDAQ']
    assert your_text_clean['ticker'].iloc[:4].tolist() == ['JPM', 'JPM', 'NVDA', 'NVDA']
    assert your_text_clean['company_name'].iloc[:4].tolist() == ['JPMORGAN CHASE', 'JPMORGAN CHASE', 'NVIDIA CORP.', 'NVIDIA CORP.']
    assert your_text_clean.loc[4, ['exchange', 'ticker', 'company_name']].isna().all(), 'Keep missing text missing.'
    assert pd.api.types.is_numeric_dtype(your_text_clean['price']), 'The parsed prices must be numeric.'
    np.testing.assert_allclose(your_text_clean['price'].iloc[:3].to_numpy(dtype=float), [210.5, 211, 0])
    assert your_text_clean['price'].iloc[3:].isna().all(), 'An unrelated number is not a valid quote.'
    assert your_text_clean['price_needs_review'].tolist() == [False, False, True, True, True]
    assert your_text_note.strip(), 'Explain what a flagged value means.'
    print('Text-cleaning checks passed. Explain why a numeric zero and a failed parse both need review.')